# Forager and farmer SPDs in the Levant and northern Levant (XRONOS)

The notebook documents the forager and farmer summed probability
distributions (SPDs) used as the empirical target of the Levantine
Neolithic agent-based model. It loads the precomputed outputs of
`xronos_SPD_levant.py` and presents them section by section. To
regenerate the underlying data, run that script with
`data_2025-02-11.csv` in the working directory.

## Contents

1. [Introduction](#Section-1--Introduction)
2. [Setup](#Section-2--Setup)
3. [Data source: XRONOS](#Section-3--Data-source-XRONOS)
4. [Methods](#Section-4--Methods)
5. [Results: forager and farmer SPDs](#Section-5--Results)
6. [Sensitivity: bounding box](#Section-6--Bounding-box)
7. [Sensitivity: classification patterns](#Section-7--Classification)
8. [Sensitivity: Cyprus exclusion](#Section-8--Cyprus)
9. [Material composition](#Section-9--Material)
10. [Geographic distribution](#Section-10--Geographic)
11. [Scope of the analysis](#Section-11--Scope)
12. [Research-intensity correction](#Section-12--Research-intensity)
13. [Limitations](#Section-13--Limitations)
14. [Handoff to the agent-based model](#Section-14--Handoff)
15. [Reference table](#Section-15--Reference-table)

## Section 1 — Introduction

### Purpose

This notebook builds a summed probability distribution (SPD) of
radiocarbon dates for Epipalaeolithic foragers and Neolithic
farmers in the Levant and northern Levant. The two curves are the
empirical target against which the Levantine Neolithic agent-based
model is calibrated.

Two features of the empirical record are of interest:

1. The **crossover age** at which the farmer dated record overtakes
   the forager dated record. This is a proxy for the timing of the
   transition.
2. The **shape** of the two curves. The forager curve peaks and
   then declines; the farmer curve rises, has two peaks, and
   declines.

### Region

The region covers the Levant corridor and the northern Levant:
lat 28–40°N, lon 30–42°E. This includes Israel, Palestine,
Jordan, Lebanon, Syria, and the upper Euphrates and Tigris
drainages of southeastern Turkey. It contains the Natufian
heartland of the southern Levant and the earliest Pre-Pottery
Neolithic sites of the northern Levant (Tell Qaramel, Jerf el
Ahmar, Göbekli Tepe, Çayönü).

### Time window

23,000–7,300 cal BP. This covers the Epipalaeolithic forager
cultures, the Natufian–PPNA transition, and the entire
Pre-Pottery Neolithic sequence.

## Section 2 — Setup

In [ ]:
%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

plt.rcParams["figure.dpi"] = 110
plt.rcParams["savefig.dpi"] = 150

Path("figures").mkdir(exist_ok=True)

expected = [
    "levant_forager_spd.npz",
    "levant_farmer_spd.npz",
    "levant_grid_corrected.npz",
    "levant_foragers.csv",
    "levant_farmers.csv",
    "levant_raw_density.csv",
]

print("Expected files:")
for f in expected:
    p = Path(f)
    print(f"  {'OK ' if p.exists() else 'MISSING'} {f}")

## Section 3 — Data source: XRONOS

XRONOS is an open-access radiocarbon database that aggregates
dates from EUROEVOL, RADON, and many regional compilations into
a single standardized schema. It is distributed as a single CSV
file on Zenodo:

    https://zenodo.org/records/14850157

Relevant fields:

  - `bp`, `std`                 raw radiocarbon age and error
  - `lat`, `lng`                site coordinates
  - `site`, `country`           site identity and country
  - `material`                  dated material
  - `species`                   species identification
  - `periods`                   broad period (JSON-like string)
  - `typochronological_units`   finer culture (JSON-like string)

Substring searches on `periods` and `typochronological_units`
provide the classification into forager versus farmer. The
Levantine corpus is dominated by the NERD dataset (Near East
Radiocarbon Dates), one of XRONOS's regional ingests.

The Levantine metadata is coarse. XRONOS's own assessment states
that non-radiocarbon chronological information in West Asia is
limited to "rough typological determinations". Classification by
substring search therefore carries uncertainty that is larger
than in better-documented regions.

## Section 4 — Methods

### Region

Bounding box: lat 28–40°N, lon 30–42°E. This covers:

  - Southern Levant: Israel, Palestine, Jordan, Lebanon
  - Central Levant: Syria, western Iraq
  - Northern Levant and Upper Mesopotamia: southeastern Turkey,
    the upper Euphrates and Tigris drainages
  - The Urfa–Diyarbakır region (Göbekli Tepe, Çayönü, Hallan Çemi)

The northern extension is necessary because the earliest PPNA
sites (Tell Qaramel, Jerf el Ahmar, Göbekli Tepe, Çayönü) lie
north of the southern-Levant bounding box. Including them
strengthens the early farmer record.

### Cyprus

Cyprus is removed via a dedicated lat/lon mask
(lat 34.5–35.7, lon 32.2–34.6). The Cypriot Neolithic is a
colonisation event, not a forager-to-farmer transition. Including
it would add ~250 farmer dates and essentially zero forager
dates, shifting the crossover and the farmer peak younger.

### Time window

23,000–7,300 cal BP. This covers the Epipalaeolithic (Kebaran,
Geometric Kebaran, Natufian, Harifian), the transition, and the
entire Pre-Pottery Neolithic sequence.

### Calibration

IntCal20 via a precomputed lookup table.

### Classification

Two classes only.

**Foragers.** Kebaran, Geometric Kebaran, Nizzanan, Nebekian,
Mushabian, Ramonian, Natufian, Harifian, generic Epipalaeolithic
labels, Late Palaeolithic, Zarzian.

**Farmers.** Pre-Pottery Neolithic A, B, C; Khiamian; Sultanian;
Mureybetian; Aswadian; generic Neolithic; Pottery Neolithic;
Yarmukian; Jericho IX; Wadi Rabah; Lodian; Halaf; Samarra;
Hassuna; Chalcolithic.

Two classification rules:

1. **Double-match resolution.** Rows that match both patterns are
   resolved to forager. This handles compound period strings at
   multi-period sites (e.g. a site with both Natufian and PPNB
   occupation).
2. **Farming horizon constraint.** Any date classified as farmer
   but calibrated older than 11,650 BP is dropped. Farming did
   not exist in the Levant before the PPNA horizon, so such
   dates are site-level attribution errors.

A transition category was considered but not used. The Levantine
metadata does not carry enough transitional labels to build one
— only ~16 raw dates matched, and 3 after thinning. That is not
an SPD.

### Thinning

One date per site per 200-year bin, chosen at random.

### SPD construction

Histogram of thinned dates into 10-year bins, followed by a
Gaussian filter with a 60-year bandwidth.

### Bootstrap confidence bands

500 resamples with replacement, 5th and 95th percentiles.

## Section 5 — Results: forager and farmer SPDs

The two SPDs are loaded from the precomputed NPZ files. The
forager curve peaks in the Late Natufian; the farmer curve has
two peaks corresponding to the PPNA / Early PPNB and Middle PPNB.

In [ ]:
def load_spd(path):
    p = Path(path)
    if not p.exists():
        return None
    d = np.load(p, allow_pickle=True)
    out = {"age": d["age"]}
    for key in ["median", "lo", "hi", "raw"]:
        if key in d.files:
            out[key] = d[key]
    return out


forager = load_spd("levant_forager_spd.npz")
farmer = load_spd("levant_farmer_spd.npz")

if forager is None or farmer is None:
    print("SPD NPZ files not present. Run xronos_SPD_levant.py first.")
else:
    fig, ax = plt.subplots(figsize=(11, 6))

    if "raw" in forager:
        ax.plot(forager["age"], forager["raw"],
                color="steelblue", linewidth=2.2,
                label="Foragers (raw)")
    elif "median" in forager:
        ax.plot(forager["age"], forager["median"],
                color="steelblue", linewidth=2.2,
                label="Foragers (median)")

    if "raw" in farmer:
        ax.plot(farmer["age"], farmer["raw"],
                color="firebrick", linewidth=2.2,
                label="Farmers (raw)")
    elif "median" in farmer:
        ax.plot(farmer["age"], farmer["median"],
                color="firebrick", linewidth=2.2,
                label="Farmers (median)")

    ax.axvline(11650, color="grey", linestyle=":", alpha=0.7,
               label="Farming horizon (11,650 BP)")
    ax.set_xlabel("Calibrated age (BP)")
    ax.set_ylabel("Raw SPD amplitude (dated sites per 10-yr bin)")
    ax.set_title("Forager and farmer SPDs — Levant and northern Levant")
    ax.set_xlim(23000, 7300)
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)
    plt.tight_layout()
    plt.savefig("figures/notebook_spd_raw.png", dpi=150)
    plt.show()

In [ ]:
if forager is not None and farmer is not None:
    def peak_of(spd):
        y = spd.get("raw", spd.get("median"))
        if y is None:
            return None, None
        i = int(np.argmax(y))
        return float(spd["age"][i]), float(y[i])

    fpk, fval = peak_of(forager)
    apk, aval = peak_of(farmer)

    print(f"Forager peak: {fpk:.0f} cal BP  "
          f"(amplitude {fval:.2f})")
    print(f"Farmer peak:  {apk:.0f} cal BP  "
          f"(amplitude {aval:.2f})")

    f_raw = forager.get("raw")
    a_raw = farmer.get("raw")
    if f_raw is not None and a_raw is not None:
        cross = None
        for i in range(len(forager["age"]) - 1, -1, -1):
            if (a_raw[i] > f_raw[i] and f_raw[i] > 1e-6):
                cross = float(forager["age"][i])
                break
        if cross is not None:
            print(f"Crossover (farmer > forager): "
                  f"{cross:.0f} cal BP")

## Section 6 — Sensitivity: bounding box

The Levantine bounding box was widened from an original
southern-Levant extent (lat 29–38°N, lon 32–42°E) to include
the northern Levant and Upper Mesopotamia (lat 28–40°N,
lon 30–42°E). This brings in southeastern Turkey, northern
Syria, and northern Iraq — the region containing the earliest
PPNA sites.

| Bounding box | Extent |
|---|---|
| Original | lat 29–38, lon 32–42 |
| Widened  | lat 28–40, lon 30–42 |

### Effect

The widened box adds several thousand dates, particularly farmer
dates from Turkey and Syria. The forager sample grows modestly
(the Natufian heartland is in the southern Levant, which is
inside both boxes). The farmer sample grows substantially.

The farmer peak shifts earlier because the northern PPNB sites
(Tell Qaramel at ~12,400, Jerf el Ahmar at ~11,700, Göbekli
Tepe at ~11,500) contribute to the early portion of the farmer
curve. The forager peak is stable.

The widened box is the default. The sensitivity check would be
to narrow back to lon 30–42 (drop the eastern Zagros piedmont,
which is archaeologically distinct) and confirm the peaks move
by only a few hundred years.

## Section 7 — Sensitivity: classification patterns

Three classification rules were tightened after the first runs:

1. **Farmer regex: `neolit` → `neolithic`.** The substring
   `neolit` matches `neolithic`, `neolithique`, `neolitikum`,
   but also `epi-neolithic`, `pre-neolithic`, and — most
   relevant here — compound strings like
   `[{"periode": "Neolithic"}, {"periode": "Epipalaeolithic"}]`.
   The tightened pattern requires the full word.

2. **Double-match resolution.** Rows that match both forager
   and farmer patterns are resolved to forager. Before this
   rule, a date in a compound period string inherited both
   labels.

3. **Farming horizon constraint.** Any date classified as farmer
   but calibrated older than 11,650 BP is dropped. This is a
   domain constraint, not a data-cleaning step: farming
   demonstrably did not exist in the Levant before the PPNA, so
   a "farmer" date older than that horizon is a site-level
   attribution error.

### Effect

The first two rules change the counts by a few dozen dates. The
third rule is the one that matters: it drops ~115 pre-PPNA
"farmer" rows that were producing a false shoulder in the
farmer SPD before 11,650 BP.

Before the constraint, the farmer curve rose from ~22,000 BP
with a small shoulder through the Epipalaeolithic. After the
constraint, the farmer curve is flat at zero before 11,650 and
rises sharply after. This matches the archaeological record.

## Section 8 — Sensitivity: Cyprus exclusion

Cyprus is removed via a dedicated lat/lon mask. The
justification is that Cyprus has no forager-farmer transition
in the sense under study.

### The case for exclusion

1. **No transition on Cyprus.** The island was essentially
   uninhabited until the early Holocene. The earliest attested
   occupation is Akrotiri-Aetokremnos around 12,000 BP — a
   small forager presence gone by 10,500. The main permanent
   settlement was by farmers from the mainland around
   10,500–10,000 BP (Cypro-PPNB). What happens on Cyprus is
   farmer colonisation of an empty island, not a transition
   from foragers to farmers.
2. **Asymmetric sample effect.** Including Cyprus adds ~250
   farmer dates and essentially zero forager dates. The
   forager SPD is unaffected; the farmer SPD shifts younger and
   acquires a Cypriot-specific component.
3. **Island demographic regime.** Lower genetic diversity,
   limited carrying capacity, higher extinction risk.
4. **Chronological offset.** Cypro-PPNB begins around
   10,500 BP, contemporary with the Middle PPNB, not with the
   PPNA.

### The case for inclusion

1. Cyprus was colonised from the mainland and culturally is
   part of the PPNB world.
2. For a study of the eastern Mediterranean Neolithic as a
   whole, Cyprus belongs.
3. Rectangular bounding boxes are arbitrary; excluding one
   island inside the box is a positive choice.

### The choice

For a study of the forager-farmer transition, exclusion is
preferable because Cyprus had no transition. For a study of the
Neolithic population of the eastern Mediterranean, inclusion
would be preferable. This notebook uses exclusion. The
sensitivity check would be to include Cyprus and confirm the
crossover and farmer peak shift by a few hundred years but the
qualitative shape is unchanged.

## Section 9 — Material composition

Both classes are dated predominantly on charcoal and bone. The
differences between the two classes are modest but real.

In [ ]:
foragers_csv = Path("levant_foragers.csv")
farmers_csv = Path("levant_farmers.csv")

if not (foragers_csv.exists() and farmers_csv.exists()):
    print("Forager/farmer CSVs not present. "
          "Run xronos_SPD_levant.py first.")
else:
    fdf = pd.read_csv(foragers_csv, low_memory=False)
    adf = pd.read_csv(farmers_csv, low_memory=False)

    def material_breakdown(df, label, top_n=6):
        if "material" not in df.columns:
            print(f"{label}: no material column")
            return
        m = df["material"].fillna("unknown").astype(str).str.lower()
        vc = m.value_counts(normalize=True).head(top_n) * 100
        print(f"\n{label} (n={len(df)}):")
        for k, v in vc.items():
            print(f"  {k:>30s}  {v:5.1f}%")

    material_breakdown(fdf, "Foragers")
    material_breakdown(adf, "Farmers")

## Section 10 — Geographic distribution

The two classes have different geographic centres of gravity.
The forager sample is concentrated in the Natufian heartland of
the southern Levant. The farmer sample is spread across the
whole region, with the northern Levant and Upper Mesopotamia
contributing the largest share after the bounding-box widening.

In [ ]:
if foragers_csv.exists() and farmers_csv.exists():
    def country_breakdown(df, label, top_n=10):
        if "country" not in df.columns:
            print(f"{label}: no country column")
            return
        vc = df["country"].fillna("unknown").astype(str).value_counts().head(top_n)
        print(f"\n{label} (n={len(df)}):")
        for k, v in vc.items():
            print(f"  {k:>30s}  {v:5d}")

    country_breakdown(fdf, "Foragers")
    country_breakdown(adf, "Farmers")

Note on the country field. XRONOS uses both ISO codes (`TR`,
`CY`, `IL`) and full names (`Turquie`, `Chypre`), and both
`Israel` and `Palestine` may appear under a combined label. If
reporting country totals, the duplicate labels should be
merged.

## Section 11 — Scope of the analysis

The window is 23,000–7,300 cal BP. This covers:

  - The Epipalaeolithic forager cultures (Kebaran, Geometric
    Kebaran, Natufian, Harifian)
  - The Natufian–PPNA transition at ~11,650 BP
  - The entire Pre-Pottery Neolithic sequence (PPNA, Early,
    Middle, and Late PPNB)
  - The beginning of the Pottery Neolithic at the end of the
    window

### Why the window ends at 7,300 BP

The purpose of building these SPDs is to study the forager-farmer
transition and the early centuries of farming in the Levant,
before the emergence of warfare and the rise of large states.
Those later phenomena — Chalcolithic chiefdoms, Bronze Age
urbanism, Iron Age territorial states — are separate research
questions with their own dynamics, and including them would mix
distinct demographic regimes into a single dated record.

The 23,000–7,300 BP window isolates the period the agent-based
model is designed to simulate: foragers, the transition, and the
first farming societies. Extending the window would require
additional mechanisms — state formation, urbanisation, warfare —
that are outside the scope of the current model. The window ends
at the close of the PPNB and the beginning of the Pottery
Neolithic, roughly the point at which the forager-farmer dynamic
has been resolved and the subsequent trajectory is shaped by
different forces.

## Section 12 — Research-intensity correction

The standard correction (Shennan/Timpson) divides the region
into cells, builds a per-cell SPD, normalises each to its own
peak, and averages with equal weight. This removes spatial bias:
a single heavily excavated site cannot dominate the regional
curve.

### Parameters

- Cell size: 1.0°
- Minimum dates per cell: 5

The original settings (0.5°, min 8) produced only 10 cells for
the forager class, which is too few for meaningful averaging —
the individual cell curves are all noisy, and averaging them
produces a noisier curve than the raw. The 1.0° cell size with
a 5-date minimum gives enough cells in both classes.

In [ ]:
gc = Path("levant_grid_corrected.npz")
if not gc.exists():
    print("Grid-corrected NPZ not present. "
          "Run xronos_SPD_levant.py first.")
else:
    g = np.load(gc, allow_pickle=True)
    x = g["age"]
    f_grid = g["forager"]
    a_grid = g["farmer"]
    n_f = int(g["n_cells_forager"])
    n_a = int(g["n_cells_farmer"])

    print(f"Forager cells used: {n_f}")
    print(f"Farmer cells used:  {n_a}")
    print(f"Forager corrected peak: {x[np.argmax(f_grid)]:.0f} cal BP")
    print(f"Farmer corrected peak:  {x[np.argmax(a_grid)]:.0f} cal BP")

    cross = None
    for i in range(len(x) - 1, -1, -1):
        if a_grid[i] > f_grid[i] and f_grid[i] > 1e-6:
            cross = float(x[i])
            break
    if cross is not None:
        print(f"Grid-corrected crossover: {cross:.0f} cal BP")

    fig, axs = plt.subplots(1, 2, figsize=(16, 6))

    ax = axs[0]
    if forager is not None and "raw" in forager:
        fr = forager["raw"]
        ax.plot(forager["age"], fr / fr.max(),
                color="steelblue", linewidth=1.4,
                linestyle="--", alpha=0.6, label="Foragers (raw)")
    if farmer is not None and "raw" in farmer:
        ar = farmer["raw"]
        ax.plot(farmer["age"], ar / ar.max(),
                color="firebrick", linewidth=1.4,
                linestyle="--", alpha=0.6, label="Farmers (raw)")
    ax.plot(x, f_grid, color="steelblue", linewidth=2.4,
            label=f"Foragers (corrected, {n_f} cells)")
    ax.plot(x, a_grid, color="firebrick", linewidth=2.4,
            label=f"Farmers (corrected, {n_a} cells)")
    ax.set_xlim(23000, 7300)
    ax.set_xlabel("Calibrated age (BP)")
    ax.set_ylabel("Normalised SPD")
    ax.set_title("Grid-cell correction — raw (dashed) vs corrected (solid)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8)

    ax = axs[1]
    if forager is not None and farmer is not None:
        fr = forager["raw"]
        ar = farmer["raw"]
        denom = fr + ar
        share = np.where(denom > 1e-6, ar / denom, np.nan)
        ax.plot(forager["age"], share, color="purple",
                linewidth=2.2, label="Farmer share of dated record")
        ax.axhline(0.5, color="grey", linestyle=":", alpha=0.6,
                   label="50% (equal)")
    ax.axvline(11650, color="grey", linestyle=":", alpha=0.7)
    ax.set_xlim(23000, 7300)
    ax.set_ylim(0, 1.05)
    ax.set_xlabel("Calibrated age (BP)")
    ax.set_ylabel("Farmer share of dated record")
    ax.set_title("Relative population proportions")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=9)

    plt.tight_layout()
    plt.savefig("figures/notebook_spd_corrected.png", dpi=150)
    plt.show()

### What the correction shows

The forager peak is stable across raw and corrected — the
Natufian florescence at ~12,720–12,760 BP is not a spatial
artefact. Whatever the 13 forager cells are, they all peak at
essentially the same time.

The farmer peak shifts meaningfully. The raw curve peaks at
~9,460 BP. The corrected curve peaks at ~9,050 BP — a 400-year
shift earlier. The corrected farmer curve has two peaks: one at
~10,300 BP (PPNA / Early PPNB) and one at ~9,050 BP (Middle
PPNB), with a trough between them at ~9,800 BP. The raw curve
merged these into a single broad peak. The corrected curve
resolves the two PPNB phases that the archaeological record
documents.

### What the correction does and does not fix

**Corrects.** Spatial excavation bias. If one heavily excavated
site contributes 200 dates, its local peak is confined to one
cell and cannot dominate the equal-weight average.

**Does not correct.** Temporal excavation bias. If the PPNB is
more heavily excavated than the PPNA across the whole region,
the correction does not help. That is a different problem and
requires external data on excavations per period, which XRONOS
does not carry. The 23,000–7,300 BP window limits the exposure
to this bias because the densely excavated later periods are
outside it.

## Section 13 — Limitations

 ### Classification depends on excavator labels

 Classification is by substring search on the `periods` and
 `typochronological_units` fields, which are strings written by
 the original excavators and compilers. The analysis does not
 independently verify those labels through material, species, or
 stratigraphic evidence.

 ### Levantine chronology is not fully standardised

 "PPNA" and "Khiamian" overlap in the literature. "Late Natufian"
 is used differently by different authors. Regional sequences
 diverge, and the same assemblage may be labelled differently by
 different excavators. Classification by substring search
 therefore carries uncertainty.

 ### Sample size is small

 The Levantine corpus of radiocarbon-dated Epipalaeolithic and
 Early Neolithic sites is limited. Bootstrap bands are wide as a
 result. The forager sample in particular is spatially
 concentrated.

 ### Metadata quality is uneven

 XRONOS's own assessment states that while the radiocarbon corpus
 for West Asia is fairly comprehensive, metadata quality could be
 substantially improved with regional expertise. Non-radiocarbon
 chronological information is limited to rough typological
 determinations. Period labels may be inconsistent across sites
 and publications.

 ### Site-level attribution errors

 Some rows inherit a period label from the site rather than from
 the specific dated context. This is common in multi-period tells
 where a deep sounding can produce a much older radiocarbon date
 that is nevertheless tagged with the site's dominant period. The
 farming-horizon constraint removes the specific case of pre-PPNA
 "farmer" dates, but the general problem is not eliminated.

 ### SPDs as a population proxy

An SPD measures the frequency of dated contexts, not the number of people. 
Using it as a population proxy assumes datable material is proportional to
population — plausible at first order, but weak enough that the curve
should be read as a relative trend, not a count. It is defensible within 
a single region and classification scheme, over a window with moderate 
taphonomic and research bias. It is not defensible as an absolute count 
or a cross-region comparison.

 ### Transition category not usable

 A transition-context category was considered but the Levantine
 metadata does not use the corresponding labels. Only ~16 raw
 dates and 3 thinned dates matched, which is not enough for an
 SPD. The Levantine result rests on the forager and farmer curves
 alone.

 ### Research-intensity correction is spatial only

 The grid-cell correction removes spatial overrepresentation of
 heavily excavated sites. It does not correct for temporal
 differences in excavation intensity. The 23,000–7,300 BP window
 limits the exposure to temporal bias.

 ### Corrections not applied

   - Taphonomic correction for differential preservation of
     charcoal and bone.

## Section 14 — Reference table

### Empirical values from the precomputed NPZ files

| Quantity | Value |
|---|---|
| Forager file | levant_forager_spd.npz |
| Farmer file | levant_farmer_spd.npz |
| Grid-corrected file | levant_grid_corrected.npz |
| Time axis range | 7,300–23,000 cal BP |
| Forager raw peak | 12,720 cal BP |
| Forager corrected peak | 12,760 cal BP |
| Farmer raw peak | 9,460 cal BP |
| Farmer corrected peaks | 10,300 and 9,050 cal BP |
| Crossover | ~11,700 cal BP |
| Farming horizon constraint | 11,650 cal BP |
| Forager cells used | 13 |
| Farmer cells used | 33 |

### Files

| File | Contents |
|---|---|
| `levant_forager_spd.npz` | arrays `age`, `median`, `lo`, `hi`, `raw` |
| `levant_farmer_spd.npz`  | arrays `age`, `median`, `lo`, `hi`, `raw` |
| `levant_grid_corrected.npz` | arrays `age`, `forager`, `farmer`, `n_cells_forager`, `n_cells_farmer` |
| `levant_foragers.csv` | thinned forager date table |
| `levant_farmers.csv`  | thinned farmer date table |
| `levant_raw_density.csv` | raw and relative density table at key ages |

### Region and time

| Quantity | Value |
|---|---|
| Bounding box | lat 28–40°N, lon 30–42°E |
| Cyprus | excluded |
| Time window | 23,000–7,300 cal BP |
| Calibration curve | IntCal20 |
| SPD bandwidth | 60 years |
| Bootstrap resamples | 500 |
| Grid cell size | 1.0° |
| Min dates per cell | 5 |